# Demo AI Chatbot (Groq)
Notebook ini mendemokan chatbot sederhana menggunakan API Groq.

## 1) Install Dependency
Jalankan sekali jika package belum terpasang.

In [1]:
# %pip install -q openai

## 2) Set API Key
Simpan key ke environment variable `GROQ_API_KEY` agar tidak hardcode di notebook.

In [1]:
import os
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Masukkan GROQ_API_KEY: ")

print("GROQ_API_KEY siap dipakai.")

GROQ_API_KEY siap dipakai.


## 3) Inisialisasi Client dan Fungsi Chatbot

In [2]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

SYSTEM_PROMPT = "Kamu adalah asisten pembelajaran data science yang ramah dan ringkas."

def ask_chatbot(user_message: str, model: str | None = None) -> str:
    preferred_models = [
        "llama-3.3-70b-versatile",
        "llama-3.1-8b-instant",
        "meta-llama/llama-4-scout-17b-16e-instruct",
        "meta-llama/llama-4-maverick-17b-128e-instruct"
    ]

    if model:
        candidate_models = [model]
    else:
        available_ids = [m.id for m in client.models.list().data]
        prioritized = [m for m in preferred_models if m in available_ids]

        include_tokens = [
            "llama", "gpt", "gemma", "mixtral", "qwen", "deepseek", "instruct", "chat", "versatile", "instant"
        ]
        exclude_tokens = [
            "whisper", "tts", "transcribe", "embed", "embedding", "rerank", "moderation", "safety", "guard", "classif"
        ]

        # Ambil model yang paling mungkin mendukung chat completions.
        remaining = [
            m for m in available_ids
            if m not in prioritized
            and any(token in m.lower() for token in include_tokens)
            and not any(token in m.lower() for token in exclude_tokens)
        ]

        # Fallback jika filter terlalu ketat: tetap hindari model non-chat umum.
        if not remaining:
            remaining = [
                m for m in available_ids
                if m not in prioritized
                and not any(token in m.lower() for token in exclude_tokens)
            ]

        candidate_models = prioritized + remaining

    if not candidate_models:
        raise RuntimeError("Tidak ada model chat yang tersedia untuk akun Groq ini.")

    last_error = None
    for m in candidate_models:
        if not m:
            continue
        try:
            response = client.chat.completions.create(
                model=m,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.3
            )
            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            error_text = str(e).lower()
            # Lanjut ke model berikutnya jika model tidak cocok untuk chat.
            if any(msg in error_text for msg in [
                "model_not_found",
                "does not exist",
                "decommissioned",
                "does not support chat completions",
                "model_terms_required",
                "requires terms acceptance",
                "text classification models",
                "single user message"
            ]):
                continue
            raise

    raise RuntimeError(f"Semua kandidat model gagal. Error terakhir: {last_error}")

## 4) Demo Tanya Jawab

In [3]:
pertanyaan = "Jelaskan perbedaan klasifikasi dan clustering dalam 3 poin singkat."
jawaban = ask_chatbot(pertanyaan)

print("Pertanyaan:")
print(pertanyaan)
print("\nJawaban Chatbot:")
print(jawaban)

Pertanyaan:
Jelaskan perbedaan klasifikasi dan clustering dalam 3 poin singkat.

Jawaban Chatbot:
**Perbedaan Klasifikasi vs. Clustering (3 poin singkat)**  

1. **Supervisi**  
   - *Klasifikasi*: Metode **terawasi** (supervised); model dilatih dengan data berlabel (contoh: “spam” atau “bukan spam”).  
   - *Clustering*: Metode **tak terawasi** (unsupervised); tidak ada label, algoritma mencari pola/kelompok secara otomatis.

2. **Tujuan Output**  
   - *Klasifikasi*: Menentukan **kelas** atau kategori tetap untuk setiap contoh baru (prediksi diskrit).  
   - *Clustering*: Mengelompokkan data ke dalam **cluster** yang mirip satu sama lain, tanpa kelas yang sudah ditentukan sebelumnya.

3. **Evaluasi**  
   - *Klasifikasi*: Dapat diukur dengan akurasi, precision, recall, F1‑score karena ada label “kebenaran”.  
   - *Clustering*: Evaluasi bersifat relatif (silhouette score, Davies‑Bouldin, atau membandingkan dengan label bila tersedia), karena tidak ada “jawaban” pasti.


## 5) Mini Chat Loop (Opsional)
Ketik `exit` untuk berhenti.

In [5]:
from IPython.display import clear_output

while True:
    user_text = input("Anda: ")
    if user_text.strip().lower() in {"exit", "quit", "keluar"}:
        clear_output(wait=True)
        print("Sesi selesai.")
        break

    clear_output(wait=True)
    print(f"Anda: {user_text}")

    try:
        bot_text = ask_chatbot(user_text)
        print("Bot:", bot_text)
    except Exception as e:
        print(f"Terjadi error saat memanggil chatbot: {e}")
        print("Tips: pastikan API key valid dan coba pertanyaan lain.")

Sesi selesai.


## 6) RAG Sederhana: Upload PDF + Tanya Jawab
Bagian ini menambahkan alur sederhana untuk:
1. Upload file PDF
2. Ekstrak isi PDF
3. Buat index TF-IDF untuk retrieval konteks
4. Tanya jawab berdasarkan isi PDF lewat tombol interaktif

Jalankan cell dari atas ke bawah.

In [ ]:
# %pip install -q pypdf pdfplumber ipywidgets scikit-learn

In [16]:
from io import BytesIO
import re
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def _clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text or "").strip()
    return text


def _chunk_text(text: str, chunk_size: int = 800, overlap: int = 120):
    words = text.split()
    if not words:
        return []

    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk_words = words[i:i + chunk_size]
        if chunk_words:
            chunks.append(" ".join(chunk_words))
    return chunks


def _extract_pdf_text_from_bytes(pdf_bytes: bytes) -> str:
    reader = PdfReader(BytesIO(pdf_bytes))
    pages = []
    for page in reader.pages:
        pages.append(_clean_text(page.extract_text()))
    return "\n".join([p for p in pages if p])


uploader = widgets.FileUpload(
    accept=".pdf",
    multiple=False,
    description="Upload PDF"
)

upload_help = widgets.HTML(
    value="<b>Langkah:</b> Klik Upload PDF, pilih 1 file, lalu klik tombol <i>Buat Index PDF</i> di cell berikutnya."
)

display(widgets.VBox([upload_help, uploader]))

In [17]:
import textwrap

PDF_CHUNKS = []
PDF_VECTORIZER = None
PDF_MATRIX = None
PDF_FILE_NAME = ""


def _read_upload_payload(upload_widget):
    value = upload_widget.value
    if not value:
        raise ValueError("Belum ada file PDF yang di-upload.")

    if isinstance(value, dict):
        item = next(iter(value.values()))
    elif isinstance(value, (tuple, list)):
        item = value[0]
    else:
        raise TypeError(f"Format upload tidak dikenali: {type(value)}")

    if isinstance(item, dict):
        file_name = item.get("name", "dokumen.pdf")
        content = item.get("content")
    else:
        file_name = getattr(item, "name", "dokumen.pdf")
        content = getattr(item, "content", None)

    if content is None:
        raise ValueError("Konten file PDF tidak ditemukan pada widget upload.")

    if isinstance(content, memoryview):
        pdf_bytes = content.tobytes()
    elif isinstance(content, bytearray):
        pdf_bytes = bytes(content)
    elif isinstance(content, bytes):
        pdf_bytes = content
    else:
        pdf_bytes = bytes(content)

    return file_name, pdf_bytes


def _is_heading_candidate(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False

    # Judul umumnya pendek dan tidak diakhiri tanda baca kalimat.
    if len(t) > 120 or len(t.split()) > 14:
        return False
    if t.endswith((".", "!", "?", ";")):
        return False

    # Pola umum judul: uppercase, title case, atau heading bernomor.
    alpha_words = [w for w in t.split() if any(ch.isalpha() for ch in w)]
    starts_upper = sum(1 for w in alpha_words if w[0].isupper())
    title_ratio = starts_upper / max(1, len(alpha_words))

    if t.isupper():
        return True
    if re.match(r"^\d+(?:\.\d+)*[\.)]?\s+", t):
        return True
    if title_ratio >= 0.7 and len(alpha_words) >= 1:
        return True

    return False


def _split_text_into_paragraphs(raw_text: str):
    normalized = (raw_text or "").replace("\r\n", "\n").replace("\r", "\n")
    raw_blocks = re.split(r"\n\s*\n+", normalized)

    # Satu block hasil ekstraksi dianggap satu paragraf.
    paragraphs = []
    for block in raw_blocks:
        lines = [line.strip() for line in block.split("\n") if line.strip()]
        if not lines:
            continue
        paragraphs.append(" ".join(lines))

    # Jika ada judul yang berdiri sendiri, gabungkan ke paragraf setelahnya.
    merged = []
    pending_titles = []

    for para in paragraphs:
        if _is_heading_candidate(para):
            pending_titles.append(para)
            continue

        if pending_titles:
            title_text = " ".join(pending_titles)
            merged.append(f"{title_text}\n{para}")
            pending_titles = []
        else:
            merged.append(para)

    # Jika dokumen berakhir dengan judul tanpa isi, tetap simpan agar tidak hilang.
    if pending_titles:
        merged.extend(pending_titles)

    return merged


def _extract_pdf_paragraphs_from_bytes(pdf_bytes: bytes):
    paragraphs = []

    # Coba parser berbasis layout dulu agar jeda paragraf lebih terjaga.
    try:
        import pdfplumber

        with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
            for page in pdf.pages:
                text = page.extract_text(layout=True) or page.extract_text() or ""
                paragraphs.extend(_split_text_into_paragraphs(text))

        if paragraphs:
            return paragraphs
    except Exception:
        # Fallback ke pypdf jika pdfplumber belum terpasang/bermasalah.
        pass

    reader = PdfReader(BytesIO(pdf_bytes))
    for page in reader.pages:
        page_text = page.extract_text() or ""
        paragraphs.extend(_split_text_into_paragraphs(page_text))
    return paragraphs


def _print_wrapped(text: str, width: int = 95, initial_indent: str = "", subsequent_indent: str = ""):
    lines = str(text).splitlines() or [str(text)]
    for line in lines:
        if not line.strip():
            print("")
            continue
        print(
            textwrap.fill(
                line,
                width=width,
                initial_indent=initial_indent,
                subsequent_indent=subsequent_indent,
                break_long_words=False,
                break_on_hyphens=False,
            )
        )


def build_pdf_index_from_upload(upload_widget=uploader):
    global PDF_CHUNKS, PDF_VECTORIZER, PDF_MATRIX, PDF_FILE_NAME

    file_name, pdf_bytes = _read_upload_payload(upload_widget)
    paragraphs = _extract_pdf_paragraphs_from_bytes(pdf_bytes)

    if not paragraphs:
        raise ValueError("Isi PDF kosong atau tidak bisa diekstrak.")

    PDF_CHUNKS = paragraphs
    PDF_VECTORIZER = TfidfVectorizer()
    PDF_MATRIX = PDF_VECTORIZER.fit_transform(PDF_CHUNKS)
    PDF_FILE_NAME = file_name

    full_text = "\n\n".join(paragraphs)
    return {
        "file_name": file_name,
        "chunks": len(PDF_CHUNKS),
        "characters": len(full_text),
    }


def ask_pdf_chatbot(question: str, top_k: int = 3):
    if not PDF_CHUNKS or PDF_VECTORIZER is None or PDF_MATRIX is None:
        raise RuntimeError("Index PDF belum dibuat. Klik 'Buat Index PDF' terlebih dahulu.")

    if not question.strip():
        raise ValueError("Pertanyaan tidak boleh kosong.")

    top_k = max(1, min(top_k, len(PDF_CHUNKS)))
    q_vec = PDF_VECTORIZER.transform([question])
    scores = cosine_similarity(q_vec, PDF_MATRIX)[0]
    top_indices = scores.argsort()[-top_k:][::-1]

    selected_paragraphs = [(int(i) + 1, PDF_CHUNKS[i], float(scores[i])) for i in top_indices]
    context = "\n\n".join([f"[Paragraf {idx}] {text}" for idx, text, _ in selected_paragraphs])

    prompt = (
        "Jawab pertanyaan pengguna hanya berdasarkan konteks PDF berikut. "
        "Jika tidak ditemukan, jawab jujur bahwa informasi tidak ada di dokumen.\n\n"
        f"Konteks:\n{context}\n\n"
        f"Pertanyaan: {question}"
    )
    answer = ask_chatbot(prompt)
    return {"answer": answer, "contexts": selected_paragraphs}


btn_build = widgets.Button(description="Buat Index PDF", button_style="info")
out_build = widgets.Output()


def on_build_clicked(_):
    with out_build:
        out_build.clear_output()
        try:
            info = build_pdf_index_from_upload(uploader)
            print(f"Index siap: {info['file_name']}")
            print(f"Jumlah chunk/paragraf: {info['chunks']} | Jumlah karakter: {info['characters']}")
            if info["chunks"] <= 5:
                print("Catatan: Struktur paragraf di PDF mungkin tidak tersimpan jelas; coba install pdfplumber untuk hasil lebih detail.")
        except Exception as e:
            print(f"Gagal membuat index: {e}")


btn_build.on_click(on_build_clicked)

question_box = widgets.Textarea(
    description="Pertanyaan",
    placeholder="Contoh: Ringkas isi dokumen ini dalam 5 poin.",
    layout=widgets.Layout(width="100%", height="90px"),
)

topk_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=8,
    step=1,
    description="Top-K",
    continuous_update=False,
)

btn_ask = widgets.Button(description="Tanya ke PDF", button_style="success")
out_answer = widgets.Output()


def on_ask_clicked(_):
    with out_answer:
        out_answer.clear_output()
        try:
            result = ask_pdf_chatbot(question_box.value, top_k=topk_slider.value)
            print(f"Dokumen: {PDF_FILE_NAME}")
            print("\nParagraf konteks terpilih:")
            for idx, text, score in result["contexts"]:
                print(f"[Paragraf {idx}] (score: {score:.4f})")
                preview = text if len(text) <= 600 else text[:600] + "..."
                _print_wrapped(preview, width=95, subsequent_indent="  ")
                print("")

            print("Jawaban:")
            _print_wrapped(result["answer"], width=95)
        except Exception as e:
            print(f"Tidak bisa menjawab: {e}")


btn_ask.on_click(on_ask_clicked)

ui = widgets.VBox([
    widgets.HTML(value="<h4>Q&A Berdasarkan PDF</h4>"),
    widgets.HBox([btn_build]),
    out_build,
    question_box,
    topk_slider,
    btn_ask,
    out_answer,
])

display(ui)

In [22]:
# Pilih chunk yang ingin dianalisis (nomor dimulai dari 1)
SELECTED_CHUNK_NUMBER = 19

if PDF_VECTORIZER is None or PDF_MATRIX is None or not PDF_CHUNKS:
    raise RuntimeError("Index PDF belum tersedia. Jalankan dulu: Upload PDF -> Buat Index PDF.")

# Validasi rentang pilihan
if SELECTED_CHUNK_NUMBER < 1 or SELECTED_CHUNK_NUMBER > len(PDF_CHUNKS):
    raise ValueError(
        f"SELECTED_CHUNK_NUMBER harus antara 1 sampai {len(PDF_CHUNKS)}, saat ini: {SELECTED_CHUNK_NUMBER}"
    )

# Variabel hasil pilihan chunk
SELECTED_CHUNK_INDEX = SELECTED_CHUNK_NUMBER - 1
SELECTED_CHUNK_TEXT = PDF_CHUNKS[SELECTED_CHUNK_INDEX]
SELECTED_CHUNK_VECTOR = PDF_MATRIX.getrow(SELECTED_CHUNK_INDEX)
SELECTED_CHUNK_NNZ = SELECTED_CHUNK_VECTOR.nnz

# Top term TF-IDF dari chunk terpilih
feature_names = PDF_VECTORIZER.get_feature_names_out()
top_n = 15
pairs = sorted(
    zip(SELECTED_CHUNK_VECTOR.indices, SELECTED_CHUNK_VECTOR.data),
    key=lambda x: x[1],
    reverse=True,
)[:top_n]
SELECTED_CHUNK_TOP_TERMS = [
    (feature_names[idx], float(weight)) for idx, weight in pairs
]

print(f"Dokumen: {PDF_FILE_NAME}")
print(f"Chunk terpilih: {SELECTED_CHUNK_NUMBER} dari {len(PDF_CHUNKS)}")
print(f"Panjang teks chunk: {len(SELECTED_CHUNK_TEXT)} karakter")
print(f"Non-zero TF-IDF: {SELECTED_CHUNK_NNZ}")

preview = SELECTED_CHUNK_TEXT[:700] + ("..." if len(SELECTED_CHUNK_TEXT) > 700 else "")
print("\nIsi chunk (preview):")
print(preview)

print("\nTop TF-IDF terms:")
if not SELECTED_CHUNK_TOP_TERMS:
    print("(kosong)")
else:
    for term, score in SELECTED_CHUNK_TOP_TERMS:
        print(f"- {term}: {score:.4f}")

Dokumen: data_analytics.pdf
Chunk terpilih: 19 dari 19
Panjang teks chunk: 389 karakter
Non-zero TF-IDF: 46

Isi chunk (preview):
Untuk kebutuhan evaluasi RAG, dokumen sumber yang rapi dan terstruktur sangat membantu. Dokumen yang memiliki judul jelas, alur logis, dan ist ilah konsisten akan meningkatkan peluang sistem retrieval menemukan konteks yang benar. Karena itu, pembuatan bahan ajar yang runtut bukan hanya berguna untuk manusia, tetapi juga meningkatkan performa pipeline AI berbasis pengetahuan organisasi.

Top TF-IDF terms:
- dokumen: 0.2528
- meningkatkan: 0.2311
- yang: 0.2252
- untuk: 0.1742
- rapi: 0.1599
- judul: 0.1599
- logis: 0.1599
- ist: 0.1599
- ilah: 0.1599
- peluang: 0.1599
- benar: 0.1599
- pembuatan: 0.1599
- bahan: 0.1599
- ajar: 0.1599
- runtut: 0.1599
